# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploration, and initial processing of the FAIR² dataset using the `mlcroissant` library. All schema elements are referenced by their `@id` as required by the Croissant specification.

### Dataset Source
The dataset is described by a Croissant schema at this URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Install the `mlcroissant` package if not already available
!pip install mlcroissant

## 1. Data Loading
Load the Croissant schema metadata and discover available record sets using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display basic metadata
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}\n")
print("Data Collection:", getattr(metadata, 'dataCollection', ''))
print("Data Biases:", getattr(metadata, 'dataBiases', ''))


## 2. Data Overview
Review available record sets, their `@id`s, and examine the fields and columns present in each.

This step is essential to identify which parts of the data to work with and how they map to actual analysis tasks.

- **Entities** such as record sets, fields, and columns are always referenced by their `@id`.

In [ ]:
from collections import defaultdict

# Discover all record set @ids from the metadata object
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print("  Fields:")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    Field @id: {field.get('@id')}, name: {field.get('name', '')}")
        else:
            print(f"    Field reference: {field}")
    print("  Columns:")
    for col in rs.get('column', []):
        if isinstance(col, dict):
            print(f"    Column @id: {col.get('@id')}, name: {col.get('name', '')}")
        else:
            print(f"    Column reference: {col}")

# For inspection: print record set ids for further steps
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"\nAll record set @ids: {record_set_ids}")

## 3. Data Extraction
Load data from each record set using its `@id` as shown above. Each record set is loaded as a DataFrame for convenient analysis. The fields and columns are referenced strictly by their `@id`.

If there is more than one record set, all are displayed. Adjust `selected_record_set_id` as needed to focus analysis below.

In [ ]:
# Load record data for each record set using their @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set {record_set_id}.")
        print(f"Columns (field @ids): {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# Use the first record set as the default for further exploration:
if len(record_set_ids) > 0:
    selected_record_set_id = record_set_ids[0]
    print(f"\nSample from record set {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
You can now manipulate data from a record set. As an example, select a numeric field for filtering and normalization using strictly its `@id` as the column reference.

- *Replace `numeric_field_id` and `group_field_id` below with actual field `@id`s from your data overview if needed.*
- The code demonstrates filtering on values greater than a threshold, mean/std normalization, and optional grouping.

**Always use `@id` for field/column references.**

In [ ]:
# Example: Select a (replaceable) numeric field and group field by their @id
df = dataframes.get(selected_record_set_id)

# Inspect field @ids to choose from (choose those shown in a previous cell)
print(f"Available columns in record set {selected_record_set_id}:")
print(list(df.columns))

# Example: Set field ids—replace these as appropriate for your dataset
numeric_field_id = None
group_field_id = None

# Try to infer a numeric field for illustration (select the first containing float/int values)
for field_id in df.columns:
    if pd.api.types.is_numeric_dtype(df[field_id]):
        numeric_field_id = field_id
        break
# Try to guess a likely group field (@id containing 'ward', 'county', 'gender', etc.)
for field_id in df.columns:
    if any(key in field_id.lower() for key in ['ward', 'county', 'gender', 'group']):
        group_field_id = field_id
        break

if not numeric_field_id:
    raise ValueError("No numeric field found—please set `numeric_field_id` manually.")

print(f"Using numeric field @id: {numeric_field_id}.")
if group_field_id:
    print(f"Using group field @id: {group_field_id}.")

# Set a threshold for the numeric field
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records in {selected_record_set_id} where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id if present
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id, as_index=False).mean(numeric_only=True)
    print(f"\nGrouped records by {group_field_id} (mean statistics):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a chosen numeric field or the relationship between numeric and group fields.
- Be sure to always reference column names via their field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=20, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Optional: boxplot or grouped bar if group field exists
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook provided a step-by-step workflow to load, explore, process, and visualize the FAIR² dataset with `mlcroissant`, using all record set and field references strictly by their Croissant `@id`.

- The dataset describes factors (from logistic regression results) associated with the adoption of indigenous and modern knowledge for rangeland management in Northern Kenya.
- All operations referenced fields and entities by `@id` for reproducibility and schema adherence.

**Next steps** could include missing data handling, advanced statistical analysis, or building models using this data in a similar pipeline.